In [3]:
import os
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [4]:
#Load Cleaned Data
df=pd.read_csv("pha_dataset_cleaned.csv")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_5760\3043961309.py:2: DtypeWarning: Columns (0: pdes, 1: name, 2: prefix) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("pha_dataset_cleaned.csv")


In [5]:
warnings.filterwarnings("ignore")

In [18]:
print("Original shape:", df.shape)

Original shape: (958524, 45)


# 2. SELECT FEATURES AND TARGET

In [19]:
feature_cols = [
    "a",
    "e",
    "i",
    "q",
    "ad",
    "n",
    "per",
    "H",
    "om",
    "w",
    "ma",
    "class"
]

target_col = "pha"

In [20]:
# Keep only required columns
df_clean = df[feature_cols + [target_col]].copy()

 3. CLEAN TARGET

In [ ]:
# Remove rows where target is missing
df_clean = df_clean.dropna(subset=[target_col])


# Convert Y/N → 1/0
df_clean[target_col] = (
    df_clean[target_col]
    .astype(str)
    .str.upper()
    .map({"Y": 1, "N": 0}))

In [22]:
# Remove rows where target conversion failed
df_clean = df_clean.dropna(subset=[target_col])


# 4. PHYSICAL SANITY CHECK

In [23]:
df_clean = df_clean[
    (df_clean["e"] >= 0) &
    (df_clean["e"] < 1.0) &
    (df_clean["a"] > 0)
].copy()

In [24]:
# 5. CREATE X AND y
# ============================================================

X = df_clean[feature_cols].copy()
y = df_clean[target_col].astype(int).copy()

print("After cleaning:", X.shape)
print("Target distribution:")
print(y.value_counts())

After cleaning: (938599, 12)
Target distribution:
pha
0    936533
1      2066
Name: count, dtype: int64


In [ ]:
# 6. TRAIN-TEST SPLIT
# ============================================================

# IMPORTANT:
# Split BEFORE learning imputation, transformation,
# outlier limits, encoding and scaling.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y)

print("\nTrain shape:", X_train.shape)
print("Test shape :", X_test.shape)


Train shape: (750879, 12)
Test shape : (187720, 12)


In [26]:
# 7. DEFINE NUMERICAL AND CATEGORICAL COLUMNS
# ============================================================

num_cols = [
    "a",
    "e",
    "i",
    "q",
    "ad",
    "n",
    "per",
    "H",
    "om",
    "w",
    "ma"
]

cat_cols = ["class"]

In [ ]:
# 8. MISSING VALUE IMPUTATION
# ============================================================

# Numerical → median
imputer = SimpleImputer(strategy="median")

X_train[num_cols] = imputer.fit_transform(X_train[num_cols])

X_test[num_cols] = imputer.transform(X_test[num_cols])


# Categorical → Unknown
X_train[cat_cols] = X_train[cat_cols].fillna("Unknown")
X_test[cat_cols] = X_test[cat_cols].fillna("Unknown")


print("\nMissing values after imputation:")
print(X_train.isnull().sum().sum())



Missing values after imputation:
0


In [ ]:
# 9. CHECK SKEWNESS
# ============================================================

print("\n================ SKEWNESS ================")

skewness = X_train[num_cols].skew()

print(skewness.sort_values(ascending=False))


In [ ]:
# 10. LOG TRANSFORMATION
# ============================================================

# Transform only strongly right-skewed POSITIVE features.
# These were identified based on the asteroid data.

skewed_cols = [ "a","q","ad","n","per"]

for col in skewed_cols:

    X_train[f"log_{col}"] = np.log1p(X_train[col].clip(lower=0))

    X_test[f"log_{col}"] = np.log1p(X_test[col].clip(lower=0))

In [29]:
# Remove original versions
X_train.drop(columns=skewed_cols, inplace=True)
X_test.drop(columns=skewed_cols, inplace=True)

In [ ]:
# 11. IQR OUTLIER CAPPING
# ============================================================

# Use TRAINING DATA to calculate IQR limits.
# Then apply the same limits to test data.

numeric_after_transform = X_train.select_dtypes(include=np.number).columns


iqr_limits = {}


for col in numeric_after_transform:

    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    iqr_limits[col] = (
        lower_bound,
        upper_bound)

    X_train[col] = X_train[col].clip(
        lower_bound,
        upper_bound)

    X_test[col] = X_test[col].clip(
        lower_bound,
        upper_bound)


print("\nIQR outlier capping completed.")


IQR outlier capping completed.


In [ ]:
# 12. ONE-HOT ENCODING
# ============================================================

ohe = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore")

train_encoded = ohe.fit_transform(X_train[cat_cols])

test_encoded = ohe.transform(X_test[cat_cols])


train_encoded_df = pd.DataFrame(
    train_encoded,
    columns=ohe.get_feature_names_out(cat_cols),
    index=X_train.index)

test_encoded_df = pd.DataFrame(
    test_encoded,
    columns=ohe.get_feature_names_out(cat_cols),
    index=X_test.index)

In [32]:
# Remove original categorical column
X_train = X_train.drop(columns=cat_cols)
X_test = X_test.drop(columns=cat_cols)

In [ ]:
# Add encoded columns
X_train = pd.concat([X_train, train_encoded_df],axis=1)

X_test = pd.concat([X_test, test_encoded_df],axis=1)


print("\nAfter One-Hot Encoding:")
print("Train:", X_train.shape)
print("Test :", X_test.shape)


After One-Hot Encoding:
Train: (750879, 23)
Test : (187720, 23)


In [ ]:
# 14. SAVE PROCESSED DATA
# ============================================================

os.makedirs("processed_data",exist_ok=True)


X_train_scaled.to_csv("processed_data/X_train.csv",index=False)

X_test_scaled.to_csv("processed_data/X_test.csv",index=False)

y_train.to_csv("processed_data/y_train.csv",index=False)

y_test.to_csv("processed_data/y_test.csv",index=False)

In [ ]:
# 15. SAVE PREPROCESSING OBJECTS
# ============================================================

joblib.dump(scaler,"processed_data/scaler.pkl")

joblib.dump(imputer,"processed_data/imputer.pkl")

joblib.dump(ohe,"processed_data/ohe.pkl")

joblib.dump(iqr_limits,"processed_data/iqr_limits.pkl")

['processed_data/iqr_limits.pkl']

In [37]:
# 16. FINAL CHECK
# ============================================================

print("\n==========================================")
print("[SUCCESS] PREPROCESSING COMPLETED")
print("==========================================")

print("Final X_train shape:", X_train_scaled.shape)
print("Final X_test shape :", X_test_scaled.shape)

print("\nSaved files:")

print(" - processed_data/X_train.csv")
print(" - processed_data/X_test.csv")
print(" - processed_data/y_train.csv")
print(" - processed_data/y_test.csv")
print(" - processed_data/scaler.pkl")
print(" - processed_data/imputer.pkl")
print(" - processed_data/ohe.pkl")
print(" - processed_data/iqr_limits.pkl")


[SUCCESS] PREPROCESSING COMPLETED
Final X_train shape: (750879, 23)
Final X_test shape : (187720, 23)

Saved files:
 - processed_data/X_train.csv
 - processed_data/X_test.csv
 - processed_data/y_train.csv
 - processed_data/y_test.csv
 - processed_data/scaler.pkl
 - processed_data/imputer.pkl
 - processed_data/ohe.pkl
 - processed_data/iqr_limits.pkl
